# 2. Preprocess an EEG-BIDS dataset

This notebook harmonises the Siena dataset converted in notebook 01. Every recording is
re-referenced to the longitudinal bipolar (double-banana) montage, notch-filtered at 50 Hz,
kept at 256 Hz and saved in HDF5 format.

The original recordings are not modified. The output is written to
`derivatives/preprocessing_<signature>/` inside the dataset, where the signature is computed
from the preprocessing settings: the same settings always give the same folder, and different
settings never overwrite each other. Running the notebook again with the same settings reuses
the recordings that are already processed.

Preprocessing Siena takes about 2 minutes on one CPU core.

In [ ]:
import json
import logging
from pathlib import Path
from time import perf_counter

import h5py
import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from IPython.display import display

from bids2seizureml.preprocessing import preprocess_bids

# Show warnings, for example when a recording is excluded.
logging.basicConfig(level=logging.WARNING, format="%(levelname)s: %(message)s")

## Settings

| Parameter | Used here | Options |
| :-- | :-- | :-- |
| `montage` | `"Longitudinal"` | `None` (keep), `"Longitudinal"`, `"Cz"` or `"Common Average"` |
| `channels` | `None` | `None` keeps all channels, or a list of names such as `["FP1-F7", "F7-T3"]` |
| `notch_filter` | `50.0` | `None`, or the mains frequency in Hz (50 Hz for Siena, 60 Hz for CHB-MIT and TUSZ) |
| `bandpass_filter` | `None` | `None`, or `(low_hz, high_hz)` |
| `target_sampling_freq` | `256.0` | `None` keeps each recording's sampling frequency |
| `output_format` | `"h5"` | `"edf"` or `"h5"` |
| `parallel`, `n_jobs` | `False`, `None` | `parallel=True` processes recordings in separate processes; `n_jobs=None` uses every CPU |
| `debug`, `debug_limit` | `False`, `2` | `debug=True` processes only the first `debug_limit` recordings, with detailed logs |

- Channel labels are standardised first: T7, T8, P7 and P8 become T3, T4, T5 and T6, also
  inside bipolar labels (F7-T7 becomes F7-T3).
- Only common-average recordings can be re-referenced (to Cz or longitudinal). A recording
  that cannot be brought to the requested montage, or that lacks a requested channel, is
  excluded with a warning.
- The operations are applied in this order: montage, channel selection, resampling, notch
  filter, bandpass filter. Filter frequencies must be below the Nyquist frequency after
  resampling (128 Hz here).
- `parallel`, `n_jobs`, `debug` and `debug_limit` change how the run is executed, not its
  output, so they do not change the output folder.

In [ ]:
DATASET_PATH = Path("../data/siena")  # A converted EEG-BIDS dataset (notebook 01).

PARAMS = {
    "montage": "Longitudinal",
    "channels": None,
    "notch_filter": 50.0,          # 60.0 for CHB-MIT and TUSZ.
    "bandpass_filter": None,
    "target_sampling_freq": 256.0,
    "output_format": "h5",
    "parallel": False,
    "n_jobs": None,
    "debug": False,
    "debug_limit": 2,
}

## Run

In [ ]:
start = perf_counter()
output_dir = preprocess_bids(DATASET_PATH, **PARAMS)
print(f"Took {(perf_counter() - start) / 60:.1f} min")

log_path = output_dir / "PREPROCESSING_LOG.json"
if not log_path.exists():
    raise RuntimeError("No recording could be preprocessed with these settings; see the warnings above.")
log = json.loads(log_path.read_text())
print(f"Output folder: derivatives/{output_dir.name}")
print(
    f"{log['num_output_files']} of {log['num_input_files']} recordings preprocessed "
    f"({log['num_created_this_run']} in this run, {log['num_skipped_existing']} already done)"
)

## Compare one recording before and after

The table compares the JSON sidecars of the original and preprocessed recording; the figure
shows a 10 s excerpt of both.

In [ ]:
SUBJECT, SESSION, RUN = "00", "01", "01"
START_S, DURATION_S = 60.0, 10.0  # Excerpt to plot, in seconds.

stem = f"sub-{SUBJECT}_ses-{SESSION}_task-szMonitoring_run-{RUN}_eeg"
folder = Path(f"sub-{SUBJECT}") / f"ses-{SESSION}" / "eeg"
original = DATASET_PATH / folder / f"{stem}.edf"
processed = output_dir / folder / f"{stem}_desc-preprocessed.{PARAMS['output_format']}"

before = json.loads(original.with_suffix(".json").read_text())
after = json.loads(processed.with_suffix(".json").read_text())
fields = ["EEGReference", "SamplingFrequency", "EEGChannelCount", "RecordingDuration", "SoftwareFilters"]
display(pd.DataFrame(
    [[before.get(field), after.get(field)] for field in fields],
    index=fields, columns=["original", "preprocessed"],
))

In [ ]:
def read_excerpt(path, start_s, duration_s):
    """Return (data in µV, channel names, sampling frequency) for a short excerpt."""
    if path.suffix == ".h5":
        with h5py.File(path, "r") as h5:
            fs = float(h5.attrs["sampling_frequency"])
            start, stop = int(start_s * fs), int((start_s + duration_s) * fs)
            data = h5["signals"][:, start:stop]
            names = [n.decode() if isinstance(n, bytes) else str(n) for n in h5["channel_names"][()]]
    else:
        raw = mne.io.read_raw_edf(path, preload=False, verbose="ERROR")
        fs = raw.info["sfreq"]
        start, stop = int(start_s * fs), int((start_s + duration_s) * fs)
        data, names = raw.get_data(start=start, stop=stop), raw.ch_names
    return data * 1e6, names, fs  # Both formats store volts.


fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, title, path in [(axes[0], "Original", original), (axes[1], "Preprocessed", processed)]:
    data, names, fs = read_excerpt(path, START_S, DURATION_S)
    time = START_S + np.arange(data.shape[1]) / fs
    spacing = 6 * np.median(data.std(axis=1)) or 1.0
    offsets = -spacing * np.arange(len(names))
    ax.plot(time, (data - np.median(data, axis=1, keepdims=True)).T + offsets, color="black", lw=0.5)
    ax.set_yticks(offsets, names)
    ax.set(title=f"{title} ({fs:g} Hz)", xlabel="Time (s)", xlim=(time[0], time[-1]))
fig.tight_layout()
plt.show()

## Other datasets

To preprocess CHB-MIT or TUSZ, change `DATASET_PATH` and set `notch_filter` to `60.0`.
CHB-MIT is already stored in the longitudinal montage, so its montage is kept as it is.
TUSZ (7,361 recordings) takes about an hour on one core; `parallel=True` spreads the
recordings over several processes.